# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã dùng chung nằm trong `day10/` (nhãn, model, huấn luyện, suy luận, reward, ảnh minh hoạ); notebook này gọi các hàm đó theo đúng thứ tự GUIDE.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU**, rồi chạy lần lượt từ trên xuống.
Adapter, dự đoán và `run.json` được lưu thẳng vào Google Drive (`MyDrive/day10_work`). Nếu Colab bị ngắt: chạy lại toàn bộ mục 0 và ô đầu của mục 1, rồi chạy lại ô đang dở. Thí nghiệm đã train xong sẽ không train lại.

## 0. Môi trường (Colab T4)

In [ ]:
REPO_URL = "https://github.com/vuthien3002-sys/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
!nvidia-smi -L
import os
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # tuỳ chọn: tải model nhanh hơn
except Exception:
    pass

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, shutil, subprocess, sys
from pathlib import Path
sys.path.insert(0, os.getcwd())
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import matplotlib.pyplot as plt
from day10 import vlm
from day10.report import draw, grid, load_preds, output_stats, summary_rows

WORK = Path("/content/drive/MyDrive/day10_work")  # adapter, dự đoán, run.json: còn nguyên khi Colab ngắt
WORK.mkdir(parents=True, exist_ok=True)
Path("report/figures").mkdir(parents=True, exist_ok=True)


def backup():
    """Chép report/ (evidence, ảnh) sang Drive."""
    shutil.copytree("report", WORK / "report", dirs_exist_ok=True)


def restore():
    if (WORK / "report").exists():
        shutil.copytree(WORK / "report", "report", dirs_exist_ok=True)


def score(name, pred, out, run=None, split="test", coord="norm1000"):
    """Gọi trọng tài scripts/score.py; meta lấy thẳng từ run.json để không gõ tay sai."""
    cmd = [sys.executable, "scripts/score.py", "--pred", str(pred), "--name", name, "--out", str(out),
           "--split", split, "--coord", coord]
    if run:
        for k in ("trainable_params", "peak_vram_gb", "train_minutes"):
            cmd += ["--meta", f"{k}={run[k]}"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout, r.stderr)
    backup()


def load_run(name):
    return json.loads((WORK / name.lower() / "run.json").read_text(encoding="utf-8"))


restore()
print("evidence đã có:", sorted(p.name for p in Path("report/evidence").glob("*.json")))

In [ ]:
!python -m pytest -q
!python -m pytest day10/tests -q

## 1. Dữ liệu: tải và chia split theo phiên quay

In [ ]:
if not Path("data/splits/meta.json").exists():
    !python scripts/prepare_data.py --kaggle

from scripts.labcommon import CLASSES, read_jsonl
splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}

In [ ]:
from day10.labels import session_overlap, split_stats

meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
print("dataset_version:", meta["dataset_version"])
for s, rows in splits.items():
    n, boxes = split_stats(rows)
    print(f"{s:5s} {n:5d} ảnh  {boxes}  sessions={len({r['session'] for r in rows})}")
overlap = session_overlap(splits)
print("session trùng giữa các split:", {k: sorted(v) for k, v in overlap.items()})
assert all(not v for v in overlap.values())

In [ ]:
samples = [r for r in splits["train"] if r["objects"]][::200][:6]
sheet = grid([draw(r) for r in samples])
sheet.save("report/figures/data_samples.jpg")
backup()
sheet

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from day10.labels import PROMPT_NORM1000, box_to_pixels, objects_to_answer, yolo_to_box, yolo_to_pixels

e = samples[0]
answer = objects_to_answer(e["objects"])
print(PROMPT_NORM1000, "\n")
print("đáp án:", answer)

# Quy ngược thang 0–1000 → pixel và so với box gốc trên toàn bộ train
err = max(abs(a - b) for r in splits["train"] for o in r["objects"]
          for a, b in zip(box_to_pixels(yolo_to_box(o["label"], o["yolo"])["bbox_2d"], r["width"], r["height"]),
                          yolo_to_pixels(o["yolo"], r["width"], r["height"])))
print(f"sai số lớn nhất khi quy ngược về pixel: {err:.2f} px")
draw(e, raw=answer)  # box đỏ (từ JSON) phải trùng box xanh (gốc)

## 3. E0: zero-shot

In [ ]:
model, processor = vlm.load_model()
print("Linear 4-bit theo khối:", vlm.check_quantization(model))  # chỉ được có 'llm'
params, _ = vlm.param_breakdown(model)
for k, v in params.items():
    print(f"{k:22s} {v / 1e6:8.1f}M")
print(f"{'tổng':22s} {sum(params.values()) / 1e6:8.1f}M")

In [ ]:
vlm.predict(model, processor, splits["test"], WORK / "e0.jsonl")
score("E0", WORK / "e0.jsonl", "report/evidence/e0.json")

In [ ]:
preds0 = load_preds(WORK / "e0.jsonl")
for e in splits["test"][:5]:
    print(e["image_id"], "\n  GT :", objects_to_answer(e["objects"]), "\n  RAW:", repr(preds0[e["image_id"]][:300]))
print(output_stats(splits["test"], preds0))

In [ ]:
del model
vlm.free()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from PIL import Image
from transformers import AutoProcessor
from day10.labels import objects_to_answer, prompt_for

processor = AutoProcessor.from_pretrained(vlm.MODEL_ID)
ex = [r for r in splits["train"] if r["objects"]][0]
img = Image.open(Path("data") / ex["image_path"]).convert("RGB")
item = vlm.encode_example(processor, img, prompt_for(), objects_to_answer(ex["objects"]))
tb = vlm.token_breakdown(processor, item)
print({k: v for k, v in tb.items() if k != "answer_decoded"})
print("phần được tính loss:", repr(tb["answer_decoded"]))
print(f"tỷ lệ token được tính loss: {tb['answer'] / tb['total']:.1%}")
assert tb["image"] == 220

batch = vlm.collate([item, item], processor.tokenizer.pad_token_id)
print({k: tuple(v.shape) for k, v in batch.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
print("LoRA LLM:", vlm.LLM_TARGETS)
print("LoRA ViT:", vlm.VIT_TARGETS)
for n in ("e1", "e2", "e3"):
    print(n, vlm.load_config(n))

# E3 chứa cả ba phần: LoRA LLM (E1), + projector (E2), + LoRA ViT (E3)
model, processor = vlm.load_model()
model, matched = vlm.apply_setup(model, vlm.load_config("e3"))
n_llm = sum("language_model" in m for m in matched)
print(f"module LoRA: LLM={n_llm} (kỳ vọng 252), ViT={len(matched) - n_llm} (kỳ vọng 96)")
by = {}
for n, p in model.named_parameters():
    if p.requires_grad:
        key = "projector" if any(k in n for k in vlm.PROJECTOR_KEYS) else ("lora_vit" if "visual" in n else "lora_llm")
        by[key] = by.get(key, 0) + p.numel()
print({k: f"{v:,}" for k, v in by.items()})
del model
vlm.free()

## 6. Vòng huấn luyện
Chạy thử 10 bước trước (khoảng 5 phút) để bắt lỗi, rồi mới chạy E1, E2, E3 (mỗi cái 300 bước + suy luận 311 ảnh test).

In [ ]:
smoke = dict(vlm.load_config("e1"), max_steps=10)
model, processor = vlm.load_model()
model, _ = vlm.apply_setup(model, smoke)
vlm.train(model, processor, splits["train"][:80], smoke, WORK / "smoke")
(WORK / "smoke_val.jsonl").unlink(missing_ok=True)
vlm.predict(model, processor, splits["val"][:8], WORK / "smoke_val.jsonl")
for r in read_jsonl(WORK / "smoke_val.jsonl"):
    print(repr(r["raw"][:200]))
del model
vlm.free()

In [ ]:
run1, pred1 = vlm.run_or_resume("E1", vlm.load_config("e1"), splits["train"], splits["test"], WORK)
score("E1", pred1, "report/evidence/e1.json", run1)

In [ ]:
run2, pred2 = vlm.run_or_resume("E2", vlm.load_config("e2"), splits["train"], splits["test"], WORK)
score("E2", pred2, "report/evidence/e2.json", run2)

In [ ]:
run3, pred3 = vlm.run_or_resume("E3", vlm.load_config("e3"), splits["train"], splits["test"], WORK)
score("E3", pred3, "report/evidence/e3.json", run3)

## 7. Suy luận và chấm E1–E3: bảng tổng hợp, ảnh minh hoạ, phân tích

In [ ]:
print("| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GB) | Phút | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |")
print("|---|---|---|---|---|---|---|---|---|---|---|")
print("\n".join(summary_rows()))

ev = {n: json.loads(Path(f"report/evidence/{n.lower()}.json").read_text(encoding="utf-8")) for n in ("E1", "E2", "E3")}
BEST = max(ev, key=lambda n: ev[n]["micro"]["f1"])
print("\nsetup có F1 cao nhất:", BEST)

In [ ]:
from day10.report import pick_examples

preds = load_preds(WORK / f"{BEST.lower()}.jsonl")
good, bad = pick_examples(splits["test"], preds)
imgs = []
for tag, group in (("good", good), ("bad", bad)):
    for i, e in enumerate(group, 1):
        im = draw(e, preds[e["image_id"]])
        im.save(f"report/figures/{BEST.lower()}_{tag}_{i}.jpg")
        imgs.append(im)
sheet = grid(imgs)
sheet.save(f"report/figures/{BEST.lower()}_examples.jpg")
backup()
sheet  # hàng trên: dự đoán đúng; hàng dưới: sai nhiều nhất. Xanh = đúng, đỏ = dự đoán

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for n in ("E1", "E2", "E3"):
    h = load_run(n)["history"]
    ax.plot([x["step"] for x in h], [x["loss"] for x in h], label=n)
ax.set_xlabel("bước tối ưu"); ax.set_ylabel("loss"); ax.legend()
fig.savefig("report/figures/loss_curves.png", dpi=120, bbox_inches="tight")
backup()

In [ ]:
# Câu 3: chấm cùng dự đoán (thang 0–1000) như thể là pixel → F1 ra sao?
(WORK / "checks").mkdir(exist_ok=True)
score(f"{BEST} chấm như pixel", WORK / f"{BEST.lower()}.jsonl", WORK / "checks" / "best_as_pixel.json", coord="pixel")

# Câu 9: kích thước box và số mẫu từng lớp
import statistics
for c in CLASSES:
    areas = [o["yolo"][2] * o["yolo"][3] for r in splits["train"] for o in r["objects"] if o["label"] == c]
    print(f"{c:8s} train boxes={len(areas):5d}  diện tích trung vị={statistics.median(areas):.2%} khung hình  "
          f"F1 test {BEST}={ev[BEST]['per_class'][c]['f1']:.3f}")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
E4 chọn biến `data_size`: 10% ảnh train so với 100%, giữ nguyên mọi thứ khác của setup tốt nhất (kể cả 300 bước). Giá trị 100% chính là setup tốt nhất đã chạy ở mục 6.

In [ ]:
from day10.labels import subset

ev = {n: json.loads(Path(f"report/evidence/{n.lower()}.json").read_text(encoding="utf-8")) for n in ("E1", "E2", "E3")}
BEST = max(ev, key=lambda n: ev[n]["micro"]["f1"])
cfg4 = vlm.load_config(BEST.lower())
train10 = subset(splits["train"], 0.1, seed=42)
print(f"E4 dựa trên {BEST}: {len(train10)} ảnh train (10%, seed=42)")

run4, pred4 = vlm.run_or_resume("E4_data_0.1", cfg4, train10, splits["test"], WORK)
score(f"E4 {BEST} data_size=0.1", pred4, "report/evidence/e4_data_0.1.json", run4)
score(f"E4 {BEST} data_size=1.0", WORK / f"{BEST.lower()}.jsonl", "report/evidence/e4_data_1.0.json", load_run(BEST))
print("\n".join(summary_rows()))

**E5 (tuỳ chọn, +10).** Chạy test reward trước, rồi GRPO 60 bước từ adapter tốt nhất, suy luận test và chấm.

In [ ]:
!python -m pytest day10/tests/test_reward.py -q

In [ ]:
from day10 import grpo

model, processor, logs = grpo.run_grpo(WORK / BEST.lower(), splits["train"], WORK / "e5", vlm.load_config("e5"))
vlm.predict(model, processor, splits["test"], WORK / "e5.jsonl")
del model
vlm.free()
score("E5", WORK / "e5.jsonl", "report/evidence/e5.json")
shutil.copy2(WORK / "e5" / "grpo_log.json", "report/e5_grpo_log.json")
backup()

In [ ]:
# Nếu Colab ngắt khi đang suy luận E5: nạp lại adapter GRPO và chạy tiếp
# model, processor = vlm.load_trained(WORK / "e5"); vlm.predict(model, processor, splits["test"], WORK / "e5.jsonl")

logs = json.loads((WORK / "e5" / "grpo_log.json").read_text(encoding="utf-8"))
steps = [l for l in logs if "reward" in l]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, key in zip(axes, ("reward", "frac_reward_zero_std", "completions/mean_length")):
    ax.plot([l["step"] for l in steps], [l[key] for l in steps]); ax.set_title(key); ax.set_xlabel("bước")
fig.savefig("report/figures/e5_grpo.png", dpi=120, bbox_inches="tight")

# Dấu hiệu reward hacking: tỷ lệ '[]', số box/ảnh, diện tích box so với SFT
for n in (BEST, "E5"):
    print(n, {k: round(v, 3) for k, v in output_stats(splits["test"], load_preds(WORK / f"{n.lower()}.jsonl")).items()})
print("\n".join(summary_rows()))
backup()

## 9. Mang kết quả về máy để commit
Tải `report.zip`, giải nén đè vào thư mục `report/` của repo trên máy, rồi commit. Đồng thời File → Download → Download .ipynb để lưu notebook đã chạy (có output) đè lên `lab_day10.ipynb`.

In [ ]:
backup()
!cd /content/K4-Track-4-Day10-VisionLanguageModel && zip -qr /content/report.zip report
from google.colab import files
files.download("/content/report.zip")